In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# --------------------------------------------------
# 1. Separate features and target
# --------------------------------------------------

target_column = "failure"

X = df.drop(columns=[target_column])
y = df[target_column]


# --------------------------------------------------
# 2. Identify column types
# --------------------------------------------------

numeric_features = [
    "oil_bbl",
    "gas_mcf",
    "water_bbl",
    "tubing_pressure",
    "runtime_hours",
]

categorical_features = [
    "field",
    "well_type",
    "status",
]


# --------------------------------------------------
# 3. Split data before preprocessing
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)


# --------------------------------------------------
# 4. Numeric preprocessing
# --------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)


# --------------------------------------------------
# 5. Categorical preprocessing
# --------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)


# --------------------------------------------------
# 6. Combine preprocessing
# --------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)


# --------------------------------------------------
# 7. Add the model
# --------------------------------------------------

full_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)


# --------------------------------------------------
# 8. Define cross-validation
# --------------------------------------------------

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# --------------------------------------------------
# 9. Define parameters to tune
# --------------------------------------------------

parameter_grid = {
    "preprocessor__numeric__imputer__strategy": [
        "mean",
        "median",
    ],
    "model__C": [
        0.01,
        0.1,
        1.0,
        10.0,
    ],
    "model__class_weight": [
        None,
        "balanced",
    ],
}


# --------------------------------------------------
# 10. Tune using only training data
# --------------------------------------------------

search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=parameter_grid,
    scoring="f1",
    cv=cross_validation,
    n_jobs=-1,
    refit=True,
)

search.fit(X_train, y_train)


# --------------------------------------------------
# 11. Retrieve the best complete pipeline
# --------------------------------------------------

best_pipeline = search.best_estimator_

print("Best parameters:", search.best_params_)
print("Best CV F1:", search.best_score_)


# --------------------------------------------------
# 12. Evaluate once on untouched test data
# --------------------------------------------------

y_pred = best_pipeline.predict(X_test)
y_probability = best_pipeline.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_probability))
